# EasyRAG Evidence Selection And Top-K For Answering

## What you will do

- retrieve more citations than the answer layer can keep
- apply `select_answer_citations()` under different budgets
- inspect which evidence survives into the answer stage

## Why this matters

Retrieval top-k and answer top-k are not the same thing. EasyRAG can retrieve several useful chunks, then keep only the subset that fits the answer budget.


## Source code anchors

- `easyrag.rag.generation.selection.select_answer_citations`
- `easyrag.rag.types.AnswerParam`
- `easyrag.rag.types.QueryResult`


In [ ]:
# ruff: noqa: E402,F401,F811
import sys
from pathlib import Path

for candidate in [Path.cwd(), *Path.cwd().parents]:
    if (candidate / "easyrag").exists():
        REPO_ROOT = candidate.resolve()
        if str(REPO_ROOT) not in sys.path:
            sys.path.insert(0, str(REPO_ROOT))
        break
else:
    raise RuntimeError("Could not locate the EasyRAG repository root.")

import json
import os
import tempfile
import time
from pathlib import Path
from pprint import pprint
from unittest import mock

from easyrag.rag import AnswerParam, EasyRAG, EvalCase, QueryParam
from easyrag.support.async_utils import run_sync
from notebooks._utils import (
    managed_demo_rag,
    print_json as _print_json,
    production_backends_ready,
    provider_ready as can_use_openai_compatible_models,
    skip_message,
    stub_embedding as _stub_embedding,
    stub_query_model as _stub_query_model,
    stub_reranker as _stub_reranker,
)

from easyrag.rag.generation.selection import select_answer_citations


## Deterministic path

The retrieval step intentionally returns more evidence than the answer layer can keep. That makes the answer-side budget visible.


In [ ]:
selection_tmp = tempfile.TemporaryDirectory()
selection_rag = EasyRAG(
    working_dir=selection_tmp.name,
    workspace="selection-demo",
    embedding_func=_stub_embedding,
    query_model_func=_stub_query_model,
)
run_sync(selection_rag.initialize_storages())
run_sync(
    selection_rag.ainsert(
        [
            "# Retrieval\nEasyRAG uses grounded retrieval and query rewrite to keep evidence traceable.\n",
            "# Storage\nRetrieved evidence is packed before answer synthesis and storage review.\n",
            "# Policy\nCitation-aware answers expose the evidence budget directly to readers.\n",
        ],
        ids=["doc::retrieval", "doc::storage", "doc::policy"],
        file_paths=["docs/retrieval.md", "docs/storage.md", "docs/policy.md"],
    )
)
selection_result = run_sync(
    selection_rag.aquery(
        "How does EasyRAG keep answers grounded?",
        QueryParam(mode="hybrid", chunk_top_k=5),
    )
)

print("Raw citations")
_print_json(selection_result.citations)


## Output inspection

The next cell applies two different answer budgets. Watch which citations disappear and why. The selector enforces both an item budget and a character budget.


In [ ]:
tight_param = AnswerParam(max_citations=2, max_context_chars=120)
loose_param = AnswerParam(max_citations=4, max_context_chars=320)

tight_selection = select_answer_citations(
    selection_result.citations,
    max_items=tight_param.max_citations,
    max_chars=tight_param.max_context_chars,
)
loose_selection = select_answer_citations(
    selection_result.citations,
    max_items=loose_param.max_citations,
    max_chars=loose_param.max_context_chars,
)

print("Tight answer budget")
_print_json(tight_selection)
print("\nLoose answer budget")
_print_json(loose_selection)
print("\nDropped under the tight budget")
dropped_locations = [citation["location"] for citation in selection_result.citations if citation not in tight_selection]
pprint(dropped_locations)


## Provider-backed path

When provider-backed retrieval is available, the same evidence-selection logic still runs locally after retrieval. The cell below keeps that boundary visible.


In [ ]:
if not can_use_openai_compatible_models():
    print("Skipping provider-backed path because OPENAI-compatible config is not set.")
else:
    provider_tmp = tempfile.TemporaryDirectory()
    provider_rag = EasyRAG(working_dir=provider_tmp.name, workspace="provider-selection-demo")
    try:
        run_sync(provider_rag.initialize_storages())
        run_sync(
            provider_rag.ainsert(
                [
                    "# Retrieval\nGrounded retrieval feeds answer selection.\n",
                    "# Policy\nCitation-aware outputs preserve traceability.\n",
                ],
                ids=["doc::retrieval", "doc::policy"],
                file_paths=["docs/retrieval.md", "docs/policy.md"],
            )
        )
        provider_result = run_sync(provider_rag.aquery("How does EasyRAG preserve traceability?", QueryParam(mode="hybrid", chunk_top_k=3)))
        provider_selection = select_answer_citations(provider_result.citations, max_items=2, max_chars=160)
        _print_json(provider_selection)
    except Exception as exc:
        print(f"Provider-backed path was skipped due to runtime error: {exc}")
    finally:
        try:
            run_sync(provider_rag.finalize_storages())
        finally:
            provider_tmp.cleanup()


## What changed and why

The answer layer keeps its own evidence budget because not every retrieved citation is worth showing to the generation prompt. The selected set should still be grounded, but it needs to be compact enough that prompting remains readable and cost-aware.


In [ ]:
run_sync(selection_rag.finalize_storages())
selection_tmp.cleanup()
print("Cleaned up the deterministic evidence-selection workspace.")


## Next steps

- Continue with [05_03_context_assembly_and_packing.ipynb](05_03_context_assembly_and_packing.ipynb) to see how the selected citations become prompt-ready context.
- Read [05-generation-overview.md](../../docs/05-generation-overview.md) for the stage-level map of evidence selection, packing, and answer synthesis.
- Compare this notebook with [05_01_query_result_to_answer.ipynb](05_01_query_result_to_answer.ipynb) if you want the broader retrieval-to-answer handoff again.
